# Análise de Dados - NovaShop

Este notebook resolve as 6 perguntas do case. Optei por começar pela pergunta 6 (qualidade dos dados) antes das análises, já que alguns campos com nulo afetam diretamente os cálculos das outras questões. Depois sigo a ordem original.

Os dados estão nos arquivos CSV fornecidos. Usei pandas para manipulação, plotly para os gráficos e scipy para o teste estatístico da pergunta 3.


## Carregamento dos dados

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pedidos    = pd.read_csv('pedidos.csv',       parse_dates=['data_pedido'])
clientes   = pd.read_csv('clientes.csv',      parse_dates=['data_cadastro'])
produtos   = pd.read_csv('produtos.csv')
itens      = pd.read_csv('itens_pedido.csv')
avaliacoes = pd.read_csv('avaliacoes.csv',    parse_dates=['data_avaliacao'])
tickets    = pd.read_csv('tickets_suporte.csv', parse_dates=['data_abertura', 'data_resolucao'])

for nome, df in [('pedidos', pedidos), ('clientes', clientes), ('produtos', produtos),
                 ('itens_pedido', itens), ('avaliacoes', avaliacoes), ('tickets_suporte', tickets)]:
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")


pedidos: 15000 linhas, 7 colunas
clientes: 3000 linhas, 8 colunas
produtos: 200 linhas, 7 colunas
itens_pedido: 36740 linhas, 6 colunas
avaliacoes: 8000 linhas, 7 colunas
tickets_suporte: 4000 linhas, 7 colunas


---
## Pergunta 6 - Inconsistências na base

Antes de qualquer análise, mapeei os campos nulos e avaliei o que fazer com cada um.


In [2]:
# visão geral de nulos por tabela
for nome, df in [('pedidos', pedidos), ('clientes', clientes), ('produtos', produtos),
                 ('itens_pedido', itens), ('avaliacoes', avaliacoes), ('tickets_suporte', tickets)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print(f"{nome}:")
        for col, n in nulos.items():
            print(f"  {col}: {n} nulos ({n/len(df)*100:.1f}%)")
    else:
        print(f"{nome}: sem nulos")


pedidos:
  valor_total: 79 nulos (0.5%)
clientes: sem nulos
produtos: sem nulos
itens_pedido:
  desconto_aplicado: 301 nulos (0.8%)
avaliacoes:
  comentario: 1679 nulos (21.0%)
tickets_suporte:
  data_resolucao: 1653 nulos (41.3%)


Achei nulos em quatro campos. Abaixo explico a decisão para cada um.

**pedidos.valor_total : 79 nulos (0,5%)**
Poucos registros. Imputei com a mediana do mesmo canal de venda, o que preserva a distribuição por canal sem distorcer o ticket médio.

**itens_pedido.desconto_aplicado : 301 nulos (0,8%)**
Interpretei como ausência de desconto e preenchi com 0. Nulo aqui provavelmente significa que o campo não foi preenchido no sistema quando nenhum cupom foi aplicado.

**avaliacoes.comentario : 1.679 nulos (21%)**
Mantive como NaN. Avaliação sem comentário é um dado válido, e o cliente deu nota mas não quis escrever nada. Criei uma flag binária `tem_comentario` para usar em análises futuras se necessário.

**tickets_suporte.data_resolucao : 1.653 nulos (41,3%)**
Também mantive como NaN. Ticket sem data de resolução significa que ainda está aberto ou escalado. Criei a coluna `tempo_resolucao_dias` para os que foram resolvidos, e uma flag `resolvido`.


In [3]:
# aplicando os tratamentos
mediana_por_canal = pedidos.groupby('canal_venda')['valor_total'].transform('median')
pedidos['valor_total'] = pedidos['valor_total'].fillna(mediana_por_canal)

itens['desconto_aplicado'] = itens['desconto_aplicado'].fillna(0.0)

avaliacoes['tem_comentario'] = avaliacoes['comentario'].notna()

tickets['resolvido'] = tickets['data_resolucao'].notna()
tickets['tempo_resolucao_dias'] = (tickets['data_resolucao'] - tickets['data_abertura']).dt.days

print("tratamentos aplicados.")


tratamentos aplicados.


In [4]:
# checagens adicionais de integridade referencial
orfaos = pedidos[~pedidos['cliente_id'].isin(clientes['id'])]
print(f"pedidos com cliente_id sem cadastro: {len(orfaos)}")

orfaos_itens = itens[~itens['pedido_id'].isin(pedidos['id'])]
print(f"itens sem pedido correspondente: {len(orfaos_itens)}")

print(f"pedidos com valor_total <= 0: {(pedidos['valor_total'] <= 0).sum()}")
print(f"itens com quantidade <= 0: {(itens['quantidade'] <= 0).sum()}")

datas_fora = pedidos[
    (pedidos['data_pedido'].dt.year < 2023) |
    (pedidos['data_pedido'].dt.year > 2024)
]
print(f"pedidos com data fora do período 2023-2024: {len(datas_fora)}")


pedidos com cliente_id sem cadastro: 0
itens sem pedido correspondente: 0
pedidos com valor_total <= 0: 0
itens com quantidade <= 0: 0
pedidos com data fora do período 2023-2024: 0


Sem problemas de integridade referencial nem valores impossíveis. A base está consistente além dos nulos já tratados.

Uma observação sobre os tickets: dos 4.000 registros, 806 estão com status `aberto` e 847 como `escalado`, totalizando 1.653 sem resolução. As categorias mais frequentes de problema são atraso (961), dúvida (945) e troca (759). Esse volume de tickets em aberto é relevante para entender a dor operacional da empresa.


---
## Pergunta 1 - Volume de pedidos por status

Calculei a distribuição e plotei em barra e pizza para facilitar a leitura.


In [5]:
status_dist = pedidos['status'].value_counts().reset_index()
status_dist.columns = ['status', 'quantidade']
status_dist['percentual'] = (status_dist['quantidade'] / len(pedidos) * 100).round(2)

print(f"{'status':<15} {'quantidade':>10} {'percentual':>12}")
print("-" * 40)
for _, row in status_dist.iterrows():
    print(f"{row['status']:<15} {row['quantidade']:>10,} {row['percentual']:>11.1f}%")
print("-" * 40)
print(f"{'total':<15} {len(pedidos):>10,} {'100.0%':>12}")

taxa_problema = status_dist[status_dist['status'].isin(['cancelado','devolvido'])]['percentual'].sum()
print(f"\ncancelado + devolvido: {taxa_problema:.1f}% dos pedidos")


status          quantidade   percentual
----------------------------------------
entregue             9,959        66.4%
cancelado            2,537        16.9%
em_transito          1,371         9.1%
devolvido            1,133         7.5%
----------------------------------------
total               15,000       100.0%

cancelado + devolvido: 24.5% dos pedidos


In [6]:
cores = {
    'entregue':    '#1B4F72',
    'em_transito': '#2E86C1',
    'cancelado':   '#C0392B',
    'devolvido':   '#F39C12',
}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    subplot_titles=('Volume absoluto', 'Distribuição percentual')
)

fig.add_trace(go.Bar(
    x=status_dist['status'],
    y=status_dist['quantidade'],
    text=status_dist['quantidade'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    marker_color=[cores.get(s, '#7F8C8D') for s in status_dist['status']],
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=status_dist['status'],
    values=status_dist['quantidade'],
    marker_colors=[cores.get(s, '#7F8C8D') for s in status_dist['status']],
    textinfo='label+percent',
    hole=0.3,
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Distribuição de pedidos por status',
    plot_bgcolor='white', paper_bgcolor='white',
    height=420, margin=dict(t=70, b=40)
)
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE', row=1, col=1)
fig.show()


A taxa combinada de cancelamento e devolução é de 24,5%, 2.537 cancelados e 1.133 devolvidos. Para um e-commerce em crescimento, esse número é alto. Em geral considera-se preocupante quando essa soma ultrapassa 15-20%, então vale investigar as causas.


---
## Pergunta 2 - Top 10 produtos mais vendidos

Usei quantidade total de itens vendidos como critério de ordenação, conforme o enunciado. A receita foi calculada como `quantidade × preço praticado × (1 - desconto aplicado)`.


In [7]:
itens['receita_item'] = (
    itens['quantidade'] * itens['preco_praticado'] * (1 - itens['desconto_aplicado'])
)

top10 = (
    itens.groupby('produto_id')
    .agg(
        qtd_total=('quantidade', 'sum'),
        receita_total=('receita_item', 'sum'),
        pedidos_distintos=('pedido_id', 'nunique')
    )
    .reset_index()
    .sort_values('qtd_total', ascending=False)
    .head(10)
    .merge(produtos[['id', 'nome', 'categoria', 'subcategoria']], left_on='produto_id', right_on='id')
)

top10['rank'] = range(1, 11)
receita_total_geral = itens['receita_item'].sum()
top10['pct_receita'] = (top10['receita_total'] / receita_total_geral * 100).round(1)

print(f"{'#':<3} {'produto':<25} {'categoria':<22} {'qtd':>6} {'receita':>16} {'% receita':>10}")
print("-" * 88)
for _, r in top10.iterrows():
    print(f"{r['rank']:<3} {r['nome']:<25} {r['categoria']:<22} {r['qtd_total']:>6,}  R$ {r['receita_total']:>12,.0f} {r['pct_receita']:>9.1f}%")

print(f"\nreceita total da base: R$ {receita_total_geral:,.0f}")
print(f"os 10 produtos mais vendidos concentram {top10['pct_receita'].sum():.1f}% da receita total")


#   produto                   categoria                 qtd          receita  % receita
----------------------------------------------------------------------------------------
1   Acessórios 43             Moda                      673  R$      787,497       0.5%
2   Tênis Esportivo 78        Esporte & Lazer           669  R$    1,895,128       1.1%
3   Tênis Esportivo 75        Esporte & Lazer           663  R$    1,117,969       0.7%
4   Notebooks 172             Eletrônicos               653  R$    1,576,158       1.0%
5   Roupas Íntimas 48         Moda                      651  R$    1,782,322       1.1%
6   Vitaminas 111             Beleza & Saúde            649  R$    1,503,830       0.9%
7   Cosméticos 92             Beleza & Saúde            642  R$    1,131,308       0.7%
8   Chás 118                  Alimentos & Bebidas       637  R$    1,647,425       1.0%
9   Suplementos 194           Beleza & Saúde            635  R$      503,030       0.3%
10  Quadros 65                C

In [8]:
top_sorted = top10.sort_values('qtd_total')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Quantidade vendida', 'Receita gerada (R$)'),
    horizontal_spacing=0.14
)

fig.add_trace(go.Bar(
    y=top_sorted['nome'], x=top_sorted['qtd_total'],
    orientation='h',
    marker_color='#1B4F72',
    text=top_sorted['qtd_total'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    y=top_sorted['nome'], x=top_sorted['receita_total'],
    orientation='h',
    marker_color='#F39C12',
    text=top_sorted['receita_total'].apply(lambda x: f'R${x/1e3:.0f}k'),
    textposition='outside',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Top 10 produtos: quantidade vs receita',
    plot_bgcolor='white', paper_bgcolor='white',
    height=480, margin=dict(t=70, b=20, l=150)
)
fig.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig.show()


Os 10 produtos mais vendidos representam apenas 8,1% da receita total, o que indica que a base não tem dependência excessiva em poucos SKUs. As categorias presentes no top 10 são Moda, Esporte & Lazer, Eletrônicos, Beleza & Saúde, Alimentos & Bebidas e Casa & Decoração.


---
## Pergunta 3 - Ticket médio B2C vs B2B

Para verificar se a diferença é significativa, usei o teste t de Welch (variâncias independentes, sem assumir homocedasticidade) com nível de significância de 5%.


In [9]:
df_seg = pedidos.merge(clientes[['id', 'segmento']], left_on='cliente_id', right_on='id')

b2c = df_seg[df_seg['segmento'] == 'B2C']['valor_total']
b2b = df_seg[df_seg['segmento'] == 'B2B']['valor_total']

print(f"{'':25} {'B2C':>12} {'B2B':>12}")
print("-" * 50)
print(f"{'n pedidos':25} {len(b2c):>12,} {len(b2b):>12,}")
print(f"{'média (R$)':25} {b2c.mean():>12,.2f} {b2b.mean():>12,.2f}")
print(f"{'mediana (R$)':25} {b2c.median():>12,.2f} {b2b.median():>12,.2f}")
print(f"{'desvio padrão (R$)':25} {b2c.std():>12,.2f} {b2b.std():>12,.2f}")


                                   B2C          B2B
--------------------------------------------------
n pedidos                       11,986        3,014
média (R$)                    1,266.17     7,751.38
mediana (R$)                  1,262.86     7,763.74
desvio padrão (R$)              712.25     4,163.38


In [10]:
t_stat, p_value = stats.ttest_ind(b2c, b2b, equal_var=False)

diff = b2b.mean() - b2c.mean()
se_diff = np.sqrt(b2c.std()**2 / len(b2c) + b2b.std()**2 / len(b2b))
ci_low  = diff - 1.96 * se_diff
ci_high = diff + 1.96 * se_diff

pooled_std = np.sqrt((b2c.std()**2 + b2b.std()**2) / 2)
cohen_d = diff / pooled_std

print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.2e}")
print(f"diferença de médias (B2B - B2C): R$ {diff:,.2f}")
print(f"IC 95% da diferença: [R$ {ci_low:,.2f}, R$ {ci_high:,.2f}]")
print(f"Cohen's d   : {cohen_d:.4f}")


t-statistic : -85.2035
p-value     : 0.00e+00
diferença de médias (B2B - B2C): R$ 6,485.21
IC 95% da diferença: [R$ 6,336.03, R$ 6,634.40]
Cohen's d   : 2.1713


O p-value é praticamente zero, então rejeito a hipótese nula a diferença entre os grupos é bem significativa. O ticket médio B2B (R$ 7.751) é cerca de 6 vezes maior que o B2C (R$ 1.266), com diferença de R$ 6.485.

O Cohen's d de 2,17 indica efeito grande, o que faz sentido dado que os desvios padrão dos dois grupos também são muito diferentes (B2B tem muito mais variação de compra). Na prática, o intervalo de confiança [R$ 6.336, R$ 6.634] mostra que mesmo no cenário mais conservador a diferença é substancial.


In [11]:
fig = go.Figure()

for seg, cor in [('B2C', '#2E86C1'), ('B2B', '#1B4F72')]:
    sub = df_seg[df_seg['segmento'] == seg]['valor_total']
    fig.add_trace(go.Violin(
        y=sub, name=seg,
        box_visible=True,
        meanline_visible=True,
        fillcolor=cor,
        opacity=0.65,
        line_color='white',
        marker=dict(color=cor)
    ))

fig.add_annotation(
    text=f"p-value < 0,001  |  Cohen's d = {cohen_d:.2f}",
    xref='paper', yref='paper', x=0.5, y=1.06,
    showarrow=False,
    font=dict(size=11, color='#555')
)

fig.update_layout(
    title='Distribuição do ticket médio por segmento',
    yaxis_title='Valor do pedido (R$)',
    plot_bgcolor='white', paper_bgcolor='white',
    height=460, margin=dict(t=80, b=40),
    violingap=0.3
)
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig.show()


---
## Pergunta 4 - Evolução mensal e sazonalidade (2023-2024)


In [12]:
pedidos['ano_mes'] = pedidos['data_pedido'].dt.to_period('M')
pedidos['mes']     = pedidos['data_pedido'].dt.month
pedidos['ano']     = pedidos['data_pedido'].dt.year

mensal = (
    pedidos.groupby('ano_mes')
    .agg(volume=('id', 'count'), receita=('valor_total', 'sum'))
    .reset_index()
)
mensal['ano_mes_str'] = mensal['ano_mes'].astype(str)
mensal['mom'] = mensal['volume'].pct_change() * 100

print(f"{'mês':<10} {'volume':>8} {'receita':>18} {'MoM%':>8}")
print("-" * 48)
for _, r in mensal.iterrows():
    mom_str = f"{r['mom']:+.1f}%" if pd.notna(r['mom']) else "   "
    print(f"{r['ano_mes_str']:<10} {r['volume']:>8,}  R$ {r['receita']:>12,.0f} {mom_str:>8}")


mês          volume            receita     MoM%
------------------------------------------------
2023-01         525  R$    1,472,355         
2023-02         492  R$    1,260,821    -6.3%
2023-03         613  R$    1,642,076   +24.6%
2023-04         564  R$    1,502,061    -8.0%
2023-05         579  R$    1,443,119    +2.7%
2023-06         531  R$    1,409,494    -8.3%
2023-07         542  R$    1,391,357    +2.1%
2023-08         584  R$    1,462,019    +7.7%
2023-09         527  R$    1,337,945    -9.8%
2023-10         554  R$    1,307,917    +5.1%
2023-11       2,317  R$    5,849,470  +318.2%
2023-12         558  R$    1,457,048   -75.9%
2024-01         542  R$    1,523,533    -2.9%
2024-02         499  R$    1,282,631    -7.9%
2024-03         551  R$    1,391,324   +10.4%
2024-04         558  R$    1,354,876    +1.3%
2024-05         608  R$    1,533,756    +9.0%
2024-06         498  R$    1,260,617   -18.1%
2024-07         588  R$    1,487,761   +18.1%
2024-08         599  R$    1,

In [13]:
# sazonalidade agregada por mês do ano
nomes_mes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

saz = (
    pedidos.groupby('mes')['id']
    .count()
    .reset_index()
    .rename(columns={'id': 'volume'})
)
saz['nome'] = nomes_mes

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        'Volume mensal de pedidos (2023-2024)',
        'Volume médio por mês do ano'
    ),
    vertical_spacing=0.2
)

fig.add_trace(go.Scatter(
    x=mensal['ano_mes_str'],
    y=mensal['volume'],
    mode='lines+markers',
    line=dict(color='#1B4F72', width=2),
    marker=dict(size=6),
    name='volume mensal'
), row=1, col=1)

cores_barra = ['#C0392B' if v == saz['volume'].max() else '#2E86C1' for v in saz['volume']]

fig.add_trace(go.Bar(
    x=saz['nome'],
    y=saz['volume'],
    marker_color=cores_barra,
    showlegend=False
), row=2, col=1)

fig.update_layout(
    title='Evolução de tempo e sazonalidade de pedidos',
    plot_bgcolor='white', paper_bgcolor='white',
    height=580, margin=dict(t=70, b=40),
    showlegend=False
)
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig.update_xaxes(tickangle=-30, row=1, col=1)
fig.show()


O padrão mais evidente é o pico de novembro, com 2.902 pedidos, quase o dobro de qualquer outro mês. Isso é consistente com a Black Friday, que concentra promoções e antecipa compras de Natal no e-commerce brasileiro.

Dezembro não se destaca como pico, o que pode indicar que os clientes compram durante a Black Friday e as entregas acontecem em dezembro (explicando talvez parte dos pedidos em trânsito). Janeiro e fevereiro são os meses mais fracos, padrão típico de pós-férias quando o orçamento familiar está mais comprometido.

Não há sazonalidade clara nos demais meses, o volume oscila entre 991 e 1.187 pedidos de janeiro a outubro, sem um ciclo definido além do pico de novembro.


---
## Pergunta 5 - Canal de aquisição vs cancelamento e valor médio

Para cruzar as bases, fiz um merge entre pedidos e clientes usando `cliente_id`.


In [14]:
df_canal = pedidos.merge(
    clientes[['id', 'canal_aquisicao', 'segmento']],
    left_on='cliente_id', right_on='id'
)

canal = (
    df_canal.groupby('canal_aquisicao')
    .agg(
        total=('id_x', 'count'),
        cancelados=('status', lambda x: (x == 'cancelado').sum()),
        devolvidos=('status', lambda x: (x == 'devolvido').sum()),
        valor_medio=('valor_total', 'mean')
    )
    .reset_index()
)

canal['tx_cancelamento'] = (canal['cancelados'] / canal['total'] * 100).round(1)
canal['tx_devolucao']    = (canal['devolvidos'] / canal['total'] * 100).round(1)

print(f"{'canal':<16} {'pedidos':>8} {'cancel%':>9} {'devol%':>8} {'ticket médio':>14}")
print("-" * 58)
for _, r in canal.sort_values('tx_cancelamento', ascending=False).iterrows():
    print(f"{r['canal_aquisicao']:<16} {r['total']:>8,} {r['tx_cancelamento']:>8.1f}% "
          f"{r['tx_devolucao']:>7.1f}%  R$ {r['valor_medio']:>8,.2f}")


canal             pedidos   cancel%   devol%   ticket médio
----------------------------------------------------------
paid_search         3,965     30.7%     6.4%  R$ 2,545.76
indicação           3,658     12.4%     8.0%  R$ 2,562.57
orgânico            3,616     11.8%     8.4%  R$ 2,515.37
redes_sociais       3,761     11.7%     7.5%  R$ 2,652.36


In [15]:
canal_sorted = canal.sort_values('tx_cancelamento', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Taxa de cancelamento e devolução por canal (%)',
        'Ticket médio por canal (R$)'
    )
)

fig.add_trace(go.Bar(
    name='cancelado',
    x=canal_sorted['canal_aquisicao'],
    y=canal_sorted['tx_cancelamento'],
    marker_color='#C0392B',
    text=canal_sorted['tx_cancelamento'].apply(lambda x: f'{x}%'),
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    name='devolvido',
    x=canal_sorted['canal_aquisicao'],
    y=canal_sorted['tx_devolucao'],
    marker_color='#F39C12',
    text=canal_sorted['tx_devolucao'].apply(lambda x: f'{x}%'),
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=canal_sorted['canal_aquisicao'],
    y=canal_sorted['valor_medio'],
    marker_color='#1B4F72',
    text=canal_sorted['valor_medio'].apply(lambda x: f'R${x:,.0f}'),
    textposition='outside',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Canal de aquisição: cancelamento, devolução e ticket médio',
    barmode='stack',
    plot_bgcolor='white', paper_bgcolor='white',
    height=440, margin=dict(t=70, b=60),
    legend=dict(orientation='h', y=-0.18)
)
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig.show()


O resultado mais relevante está no canal `paid_search`: taxa de cancelamento de 30,7%, enquanto os outros três canais ficam todos entre 11,7% e 12,4%. A taxa de devolução do paid_search (6,4%) é inclusive a menor entre os canais o problema está especificamente no cancelamento, não na devolução pós-entrega.

Uma hipótese para isso: campanhas pagas podem estar gerando expectativas que o produto ou o processo de compra não cumpre, seja por anúncios imprecisos, problemas de UX/UI, ou atração de um público menos qualificado que abandona o pedido logo após a criação. Vale investigar em que etapa do pedido os cancelamentos do paid_search ocorrem.

Quanto ao ticket médio, `redes_sociais` tem o maior valor (R$ 2.652), seguido por `indicação` (R$ 2.563). A diferença entre canais é pequena a variação mais relevante está nas taxas de problema, não no valor médio.


---
## Considerações finais

Os dois achados que mais chamam atenção na análise:

O primeiro é a taxa de cancelamento do `paid_search` (30,7% vs ~12% dos outros canais). Parece ser a principal causa raiz operacional que a NovaShop está buscando e pode estar gerando parte do crescimento dos tickets de suporte, já que cancelamentos frequentemente geram contato com o atendimento.

O segundo é o volume de tickets sem resolução: 1.653 de 4.000 (41,3%) estão abertos ou escalados. As categorias mais recorrentes são atraso e dúvida, o que dá ha entender tanto problema logístico quanto falta de informação proativa ao cliente. Em um e-commerce em expansão, isso tende a piorar conforme o volume de pedidos cresce, se não for tratado antes.
